# **Geographical population structure and Pairwise Average Fst between coastal and inland cohorts**

In [3]:
!pip install -q malariagen_data

In [2]:
!pip install scikit-learn

In [3]:
import os
import re
import plotly
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
from sklearn.cluster import KMeans
import numpy as np

In [4]:
from google.colab import drive
drive.mount("drive")

Mounted at drive


In [5]:
results_dir = 'drive/MyDrive/Colab Data/gm-analysis/pca-results'
os.makedirs(results_dir, exist_ok=True)

In [6]:
import malariagen_data

In [7]:
ag3 = malariagen_data.Ag3(results_cache=results_dir)
ag3

<MalariaGEN Ag3 API client>
Storage URL                           : gs://vo_agam_release_master_us_central1/
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15
Results cache                         : /content/drive/MyDrive/Colab Data/gm-analysis/pca-results
Cohorts analysis                      : 20250815
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.6.0
Client location                       : Oregon, United States (Google Cloud us-west1)
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15
---
Please note that data are subject to terms of use,
for more information see https://www.malariagen.net/data
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.6.0/Ag3.html

In [8]:
df_samples=ag3.sample_metadata(sample_query="country=='Gambia, The' and taxon == 'arabiensis' and unrestricted_use == True")

In [9]:
gambia_sample_query = "country=='Gambia, The' and taxon == 'arabiensis' and sample_set == '1272-VO-GM-OPONDO-VMF00160'"

In [10]:
# access metadata
df_samples_gambia = ag3.sample_metadata(sample_query=gambia_sample_query)
df_samples_gambia

,sample_id,partner_sample_id,contributor,country,location,year,month,latitude,longitude,sex_call,...,admin1_name,admin1_iso,admin2_name,taxon,cohort_admin1_year,cohort_admin1_month,cohort_admin1_quarter,cohort_admin2_year,cohort_admin2_month,cohort_admin2_quarter
0,VBS47723-6366STDY9854332,C0454,Kevin Opondo,"Gambia, The",Dongoroba,2019,9,13.380,-15.294,F,...,Mansakonko,GM-L,Jarra East,arabiensis,GM-L_arab_2019,GM-L_arab_2019_09,GM-L_arab_2019_Q3,GM-L_Jarra-East_arab_2019,GM-L_Jarra-East_arab_2019_09,GM-L_Jarra-East_arab_2019_Q3
1,VBS47725-6366STDY9854334,C2409,Kevin Opondo,"Gambia, The",Kerewan,2019,9,13.515,-14.896,F,...,Janjanbureh,GM-M,Lower Fuladu West,arabiensis,GM-M_arab_2019,GM-M_arab_2019_09,GM-M_arab_2019_Q3,GM-M_Lower-Fuladu-West_arab_2019,GM-M_Lower-Fuladu-West_arab_2019_09,GM-M_Lower-Fuladu-West_arab_2019_Q3
2,VBS47726-6366STDY9854335,C0587,Kevin Opondo,"Gambia, The",Bansang,2019,9,13.434,-14.655,F,...,Janjanbureh,GM-M,Upper Fuladu West,arabiensis,GM-M_arab_2019,GM-M_arab_2019_09,GM-M_arab_2019_Q3,GM-M_Upper-Fuladu-West_arab_2019,GM-M_Upper-Fuladu-West_arab_2019_09,GM-M_Upper-Fuladu-West_arab_2019_Q3
3,VBS47729-6366STDY9854338,C2710,Kevin Opondo,"Gambia, The",Boiram,2019,10,13.503,-14.926,F,...,Janjanbureh,GM-M,Lower Fuladu West,arabiensis,GM-M_arab_2019,GM-M_arab_2019_10,GM-M_arab_2019_Q4,GM-M_Lower-Fuladu-West_arab_2019,GM-M_Lower-Fuladu-West_arab_2019_10,GM-M_Lower-Fuladu-West_arab_2019_Q4
4,VBS47731-6366STDY9854340,C2524,Kevin Opondo,"Gambia, The",Bansang,2019,9,13.434,-14.655,F,...,Janjanbureh,GM-M,Upper Fuladu West,arabiensis,GM-M_arab_2019,GM-M_arab_2019_09,GM-M_arab_2019_Q3,GM-M_Upper-Fuladu-West_arab_2019,GM-M_Upper-Fuladu-West_arab_2019_09,GM-M_Upper-Fuladu-West_arab_2019_Q3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
310,VBS48167-6366STDY9854768,W1491,Kevin Opondo,"Gambia, The",Essau,2019,10,13.481,-16.515,F,...,Kerewan,GM-N,Lower Niumi,arabiensis,GM-N_arab_2019,GM-N_arab_2019_10,GM-N_arab_2019_Q4,GM-N_Lower-Niumi_arab_2019,GM-N_Lower-Niumi_arab_2019_10,GM-N_Lower-Niumi_arab_2019_Q4
311,VBS48171-6366STDY9854772,W2321,Kevin Opondo,"Gambia, The",Brikama-A,2019,9,13.290,-16.657,F,...,Brikama,GM-W,Kombo Central,arabiensis,GM-W_arab_2019,GM-W_arab_2019_09,GM-W_arab_2019_Q3,GM-W_Kombo-Central_arab_2019,GM-W_Kombo-Central_arab_2019_09,GM-W_Kombo-Central_arab_2019_Q3
312,VBS48173-6366STDY9854774,W0382,Kevin Opondo,"Gambia, The",Tallinding,2019,9,13.437,-16.670,F,...,Kanifing,GM-W,Jeshwang,arabiensis,GM-W_arab_2019,GM-W_arab_2019_09,GM-W_arab_2019_Q3,GM-W_Jeshwang_arab_2019,GM-W_Jeshwang_arab_2019_09,GM-W_Jeshwang_arab_2019_Q3
313,VBS48174-6366STDY9854775,W1168,Kevin Opondo,"Gambia, The",Tallinding,2019,9,13.437,-16.670,F,...,Kanifing,GM-W,Jeshwang,arabiensis,GM-W_arab_2019,GM-W_arab_2019_09,GM-W_arab_2019_Q3,GM-W_Jeshwang_arab_2019,GM-W_Jeshwang_arab_2019_09,GM-W_Jeshwang_arab_2019_Q3


In [11]:
df_samples_gambia['cohort_admin1_year'].unique()

array(['GM-L_arab_2019', 'GM-M_arab_2019', 'GM-L_arab_2001',
       'GM-U_arab_2019', 'GM-W_arab_2019', 'GM-N_arab_2019'], dtype=object)

In [12]:
df_samples_gambia.groupby(['sample_set','admin1_name','cohort_admin1_year','cohort_admin2_year']).size()

sample_set                  admin1_name  cohort_admin1_year  cohort_admin2_year              
1272-VO-GM-OPONDO-VMF00160  Basse        GM-U_arab_2019      GM-U_Basse_arab_2019                28
                                                             GM-U_Jimara_arab_2019               37
                                                             GM-U_Sandu_arab_2019                 8
                                                             GM-U_Tumana_arab_2019               21
                                                             GM-U_Wuli-East_arab_2019             1
                            Brikama      GM-W_arab_2019      GM-W_Foni-Brefet_arab_2019           1
                                                             GM-W_Kombo-Central_arab_2019        28
                            Janjanbureh  GM-M_arab_2019      GM-M_Lower-Fuladu-West_arab_2019    51
                                                             GM-M_Niamina-East_arab_2019         10
                                                             GM-M_Niamina-West_arab_2019          8
                                                             GM-M_Upper-Fuladu-West_arab_2019    12
                            Kanifing     GM-W_arab_2019      GM-W_Bakau_arab_2019                 6
                                                             GM-W_Jeshwang_arab_2019             77
                            Kerewan      GM-N_arab_2019      GM-N_Illiasa_arab_2019               3
                                                             GM-N_Lower-Niumi_arab_2019          13
                            Kuntaur      GM-M_arab_2019      GM-M_Niani_arab_2019                 4
                            Mansakonko   GM-L_arab_2001      GM-L_Jarra-East_arab_2001            1
                                         GM-L_arab_2019      GM-L_Jarra-East_arab_2019            6
dtype: int64

### **Geographical population structure**

In [13]:
# parameters to use in the PCA computations
region = "3L"
n_snps = 100_000
arab_sample_query = ("taxon == 'arabiensis' and country == 'Gambia, The' and sample_set == '1272-VO-GM-OPONDO-VMF00160'")
site_mask = "arab"

In [ ]:
#pca = PCA(n_components=20)

In [14]:
pca_df_arab, evr_arab = ag3.pca(
    region=region,
    n_snps=n_snps,
    sample_query=arab_sample_query,
    site_mask=site_mask,
    min_minor_ac=2,
    max_missing_an=0
    )

**Color by collection location**

In [15]:
ag3.plot_pca_variance(evr_arab)

In [16]:
ag3.plot_pca_coords(
    pca_df_arab,
    color='admin1_name',
    color_discrete_sequence=px.colors.qualitative.Dark24,
    title = "The Gambia, Geographical population structure in An. arabiensis"
)

In [17]:
# remove outliers
arab_outliers = [
    "VBS48175-6366STDY9854776",
    "VBS47997-6366STDY9854595",
    "VBS48073-6366STDY9854673",
    "VBS48085-6366STDY9854685",
    "VBS48029-6366STDY9854629",
    "VBS47764-6366STDY9854373",
    "VBS47995-6366STDY9854593",
    "VBS48055-6366STDY9854655",
    "VBS47767-6366STDY9854376",
    "VBS48013-6366STDY9854611",
    # "VBS48028-6366STDY9854628",
    "VBS48131-6366STDY9854733",
    "VBS47999-6366STDY9854597",
    "VBS47900-6366STDY9854498",
    "VBS47896-6366STDY9854495",
    # "VBS48133-6366STDY9854735",
    "VBS47775-6366STDY9854384",
    "VBS48126-6366STDY9854728",
    # "VBS48058-6366STDY9854658",
]

In [18]:
region = "3L"
n_snps = 100_000

pca_df_arab_wo_outliers, evr_arab_wo_outliers = ag3.pca(
    region=region,
    n_snps=n_snps,
    sample_query=f"taxon == 'arabiensis' and country == 'Gambia, The' and sample_set == '1272-VO-GM-OPONDO-VMF00160' and sample_id not in {arab_outliers}",
    site_mask="arab",
    min_minor_ac=2,
    max_missing_an=0
    )

### **Supplementary Figure 1. Proportion of genetic variance explained by the first 20 principal components**

In [19]:
ag3.plot_pca_variance(evr_arab_wo_outliers)

In [20]:
print(evr_arab_wo_outliers[0] * 100)

0.7550605


In [21]:
ag3.plot_pca_coords(
    pca_df_arab_wo_outliers,
    color='admin1_name',
    title = "The Gambia, Geographical population structure in An. arabiensis"
)

In [22]:
ag3.plot_pca_coords(
    pca_df_arab_wo_outliers,
    color='location',
    color_discrete_sequence=px.colors.qualitative.Dark24,
    title = "The Gambia, Geographical population structure in An. arabiensis"
)

**Coloring by administrative division**

In [23]:
# Define color mapping for each admin1 unit
color_mapping_admin1 = {
    'Mansakonko': '#ffd966',
    'Janjanbureh': '#f4a261',
    'Kuntaur': '#2ca25f',
    'Basse': '#99d8c9',
    'Kanifing': '#a6bddb',
    'Brikama': '#d2c4e6',
    'Kerewan': '#2b8cbe'
}

In [24]:
# Apply color mapping to the PCA DataFrame
pca_df_arab_wo_outliers['color_admin1'] = pca_df_arab_wo_outliers['admin1_name'].map(color_mapping_admin1)

In [25]:
ag3.plot_pca_coords(
    pca_df_arab_wo_outliers,
    color='admin1_name',
    color_discrete_map=color_mapping_admin1,
    width=800,
    title = "The Gambia, Geographical population structure in An. arabiensis"
)

In [26]:
# Define color mapping for each admin2 unit
color_mapping_admin2 = {
    'Jarra East': '#8e8c84',
    'Lower Fuladu West': '#7f7f7f',
    'Upper Fuladu West': '#6b705c',
    'Niani': '#4a4e4d',
    'Niamina East': '#3b5c40',
    'Jimara': '#8e7d32',
    'Basse': '#a95b29',
    'Wuli East': '#b64d3b',
    'Tumana': '#8a4b39',
    'Niamina West': '#6f4e37',
    'Sandu': '#3e3a3a',
    'Jeshwang': '#a4c9e2',
    'Kombo Central': '#78aaff',
    'Lower Niumi': '#4d88ff',
    'Bakau': '#0077b6',
    'Illiasa': '#0096c7',
    'Foni Brefet': '#00b4d8'
}

In [27]:
# Apply color mapping to the PCA DataFrame
pca_df_arab_wo_outliers['color_admin2'] = pca_df_arab_wo_outliers['admin2_name'].map(color_mapping_admin2)

In [28]:
ag3.plot_pca_coords(
    pca_df_arab_wo_outliers,
    color='admin2_name',
    color_discrete_map=color_mapping_admin2,
    title = "The Gambia, Geographical population structure in An. arabiensis"
)

In [29]:
ag3.plot_pca_coords(
    pca_df_arab_wo_outliers,
    y="PC3",
    x="PC1",
    color='admin2_name',
    color_discrete_map=color_mapping_admin2,
    title = "The Gambia, Geographical population structure in An. arabiensis"
)

**Coloring by latitude/longitude coordinates**

In [30]:
ag3.plot_pca_coords(
    pca_df_arab_wo_outliers,
    x="longitude",
    y="PC1",
    color='admin1_name',
    color_discrete_map=color_mapping_admin1,
    title = "The Gambia, Geographical population structure in An. arabiensis"
)

### **Supplementary Figure 2a. Population structure**

In [31]:
ag3.plot_pca_coords(
    pca_df_arab_wo_outliers,
    x="longitude",
    y="PC1",
    color='admin2_name',
    labels={"admin2_name": "Location"},
    color_discrete_map=color_mapping_admin2,
    title = "The Gambia, Geographical population structure in An. arabiensis"
)

**Flipping the Axis to match the geographies of The Gambia**

In [32]:
print(pca_df_arab_wo_outliers)

                    sample_id        PC1        PC2         PC3        PC4  \
0    VBS47723-6366STDY9854332 -23.419657 -10.076426  -10.242103   2.417907   
1    VBS47725-6366STDY9854334 -37.601727  19.753054  -25.079685  27.718462   
2    VBS47726-6366STDY9854335 -39.161430 -10.772594  -23.454910  19.046753   
3    VBS47729-6366STDY9854338 -34.665707   2.075543  -35.114807  15.089153   
4    VBS47731-6366STDY9854340 -32.574780  -0.562764    8.660203  23.942385   
..                        ...        ...        ...         ...        ...   
294  VBS48165-6366STDY9854766  39.832607  -4.415349  -16.095625  -1.396065   
295  VBS48167-6366STDY9854768  39.658714 -29.991095   24.566141 -29.580111   
296  VBS48171-6366STDY9854772  39.874981 -63.482452  123.014488  64.019966   
297  VBS48173-6366STDY9854774  42.604729   4.097416   -4.152700  -7.620349   
298  VBS48174-6366STDY9854775  62.740772 -37.946060   12.800014 -47.646404   

            PC5        PC6         PC7         PC8        PC9  

In [33]:
# flip PC1 to switch the clusters
pca_df_arab_wo_outliers['PC1'] = -pca_df_arab_wo_outliers['PC1']

In [34]:
pca_df_arab_wo_outliers[["PC1"]]

,PC1
0,23.419657
1,37.601727
2,39.161430
3,34.665707
4,32.574780
...,...
294,-39.832607
295,-39.658714
296,-39.874981
297,-42.604729


In [35]:
pca_fig=ag3.plot_pca_coords(
    pca_df_arab_wo_outliers,
    y="PC2",
    x="PC1",
    color='admin2_name',
    labels={"admin2_name": "Location"},
    color_discrete_map=color_mapping_admin2,
    title = "The Gambia, Geographical population structure in An. arabiensis",
    show=False
)

In [36]:
pca_fig

### **Figure 1b. Geographical population structure**

**Cluster annotation - kmeans**

In [37]:
df_samples_gambia.median_cov.min()

np.int64(10)

In [38]:
df_samples_gambia.median_cov.max()

np.int64(53)

In [39]:
df_samples_gambia.median_cov.mean()

np.float64(29.847619047619048)

In [40]:
kmeans = KMeans(n_clusters=2, init = 'k-means++', max_iter=200)

In [41]:
df_columns = ['PC1', 'PC2']

In [42]:
km = kmeans.fit(pca_df_arab_wo_outliers[df_columns])

In [43]:
centroids = km.cluster_centers_

In [44]:
# Find assignment by cluster
data_pred = kmeans.fit_predict(pca_df_arab_wo_outliers[df_columns])
pca_df_arab_wo_outliers['cluster'] = data_pred

In [45]:
# Split data by group for generated bounds for ellipse
data0 = pca_df_arab_wo_outliers[pca_df_arab_wo_outliers['cluster']==0]
data1 = pca_df_arab_wo_outliers[pca_df_arab_wo_outliers['cluster']==1]

In [46]:
# Calculate covariance matrix
cov_matrix = np.cov(data0['PC1'], data0['PC2'])
# Calculate eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
# Determine the angle from eigenvectors
angle = np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])
# Get center and scale of ellipse
center = np.mean(np.array([data0['PC1'], data0['PC2']]), axis=1)
# For a 95% confidence ellipse, the scaling factor for the radii is approx. sqrt(5.991)
# This factor is derived from the chi-squared distribution with 2 degrees of freedom for 95% confidence [7]
scaling_factor = np.sqrt(5.991)
radii = scaling_factor * np.sqrt(eigenvalues)

t = np.linspace(0, 2 * np.pi, 100)
ellipse_x_auto = center[0] + radii[0] * np.cos(t) * np.cos(angle) - radii[1] * np.sin(t) * np.sin(angle)
ellipse_y_auto = center[1] + radii[0] * np.cos(t) * np.sin(angle) + radii[1] * np.sin(t) * np.cos(angle)

In [47]:
pca_fig.add_trace(go.Scatter(
    x=ellipse_x_auto,
    y=ellipse_y_auto,
    mode='lines',
    name='95% Confidence Ellipse',
    line=dict(color='#b64d3b', width=2)
    #fill='toself' # Fills the area enclosed by the ellipse
    ))


pca_fig.show()

In [48]:
# Calculate covariance matrix
cov_matrix1 = np.cov(data1['PC1'], data1['PC2'])
# Calculate eigenvalues and eigenvectors
eigenvalues1, eigenvectors1 = np.linalg.eig(cov_matrix1)
# Determine the angle from eigenvectors
angle1 = np.arctan2(eigenvectors1[1, 0], eigenvectors1[0, 0])
# Get center and scale of ellipse
center1 = np.mean(np.array([data1['PC1'], data1['PC2']]), axis=1)
# For a 95% confidence ellipse, the scaling factor for the radii is approx. sqrt(5.991)
# This factor is derived from the chi-squared distribution with 2 degrees of freedom for 95% confidence [7]
scaling_factor1 = np.sqrt(5.991)
radii1 = scaling_factor1 * np.sqrt(eigenvalues1)

t1 = np.linspace(0, 2 * np.pi, 100)
ellipse_x_auto1 = center1[0] + radii1[0] * np.cos(t1) * np.cos(angle1) - radii1[1] * np.sin(t1) * np.sin(angle1)
ellipse_y_auto1 = center1[1] + radii1[0] * np.cos(t1) * np.sin(angle1) + radii1[1] * np.sin(t1) * np.cos(angle1)

In [49]:
pca_fig.add_trace(go.Scatter(
    x=ellipse_x_auto1,
    y=ellipse_y_auto1,
    mode='lines',
    name='95% Confidence Ellipse',
    line=dict(color='#a4c9e2', width=2),
    #fill='toself' # Fills the area enclosed by the ellipse
    ))


pca_fig.show()

In [50]:
pca_fig.add_trace(go.Scatter(
    x=centroids[:,0],
    y=centroids[:,1],
    mode="text",
    name="Centroids",
    text=["IC", "CC"],
    textposition="top center"
    ))
pca_fig.update_traces(textfont_size=20)

In [51]:
admin_labels=df_samples.admin2_name.unique()

In [52]:
for trace in pca_fig['data']:
    if (not trace['name'] in admin_labels):
        trace['showlegend'] = False

In [53]:
pca_fig

**PCA - Coastal vs Inland samples**

In [54]:
coastal_cohorts = [
    "Kerewan",
    "Brikama",
    "Kanifing"
]

In [55]:
inland_cohorts = [
    "Basse",
    "Janjanbureh",
    "Kuntaur",
    "Mansakonko"
]

In [56]:
# coastal cohorts

region = "3L"
n_snps = 100_000

pca_df_arab_coast, evr_arab_coast = ag3.pca(
    region=region,
    n_snps=n_snps,
    sample_query=f"taxon == 'arabiensis' and country == 'Gambia, The' and sample_set == '1272-VO-GM-OPONDO-VMF00160' and admin1_name in {coastal_cohorts} and sample_id not in {arab_outliers}",
    site_mask="arab",
    min_minor_ac=2,
    max_missing_an=0
    )

In [57]:
ag3.plot_pca_variance(evr_arab_coast)

In [58]:
# Apply the same color mapping to the PCA DataFrame
pca_df_arab_coast['color'] = pca_df_arab_coast['admin1_name'].map(color_mapping_admin1)
pca_df_arab_coast['color'] = pca_df_arab_coast['admin2_name'].map(color_mapping_admin2)

In [59]:
ag3.plot_pca_coords(
    pca_df_arab_coast,
    color='admin2_name',
    color_discrete_map=color_mapping_admin2,
    title = "Population structure in An. arabiensis from the coastal populations"
)

### **Supplementary Figure 2b. Neighbour joining tree**

In [60]:
ag3.plot_njt(
    region="3L",
    n_snps=50_000,
    site_mask="arab",
    sample_query=f"taxon == 'arabiensis' and country == 'Gambia, The' and sample_set == '1272-VO-GM-OPONDO-VMF00160' and sample_id not in {arab_outliers}",
    color="admin2_name",
    #labels={"admin2_name": "Location"},
    color_discrete_map=color_mapping_admin2
)

In [61]:
# inland cohorts

region = "3L"
n_snps = 100_000

pca_df_arab_inland, evr_arab_inland = ag3.pca(
    region=region,
    n_snps=n_snps,
    sample_query=f"taxon == 'arabiensis' and country == 'Gambia, The' and sample_set == '1272-VO-GM-OPONDO-VMF00160' and admin1_name in {inland_cohorts} and sample_id not in {arab_outliers}",
    site_mask="arab",
    min_minor_ac=2,
    max_missing_an=0
    )

In [62]:
ag3.plot_pca_variance(evr_arab_inland)

In [63]:
# Apply the same color mapping to the PCA DataFrame
pca_df_arab_inland['color'] = pca_df_arab_inland['admin1_name'].map(color_mapping_admin1)
pca_df_arab_inland['color'] = pca_df_arab_inland['admin2_name'].map(color_mapping_admin2)

In [64]:
ag3.plot_pca_coords(
    pca_df_arab_inland,
    color='admin2_name',
    color_discrete_map=color_mapping_admin2,
    title = "Population structure in An. arabiensis from the inland populations"
)

**Adding reference populations**

In [65]:
ref_arab_outliers = [
    "VBS48175-6366STDY9854776",
    "VBS47997-6366STDY9854595",
    "VBS48073-6366STDY9854673",
    "VBS48085-6366STDY9854685",
    "VBS48029-6366STDY9854629",
    "VBS47995-6366STDY9854593",
    "VBS48055-6366STDY9854655",
    "VBS47767-6366STDY9854376",
    "VBS48013-6366STDY9854611",
    "VBS48131-6366STDY9854733",
    "VBS47999-6366STDY9854597",
    "VBS48028-6366STDY9854628",
    "VBS48133-6366STDY9854735",
    "VBS48058-6366STDY9854658",

#     "VBS48171-6366STDY9854772", #from here
#     "VBS84508-6020STDY14423981"
    # "VBS47900-6366STDY9854498",
    # "VBS47896-6366STDY9854495",
    # "VBS47775-6366STDY9854384",
    # "VBS48126-6366STDY9854728"
]

**Add reference populations from all currently published arabiensis data (as of 12th September 2024)**

In [66]:
ag3.sample_sets(["3.0","3.4", "3.5", "3.6", "3.7", "3.8"])

,sample_set,sample_count,study_id,study_url,terms_of_use_expiry_date,terms_of_use_url,release,unrestricted_use
0,AG1000G-AO,81,AG1000G-AO,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,3.0,True
1,AG1000G-BF-A,181,AG1000G-BF-1,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,3.0,True
2,AG1000G-BF-B,102,AG1000G-BF-1,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,3.0,True
3,AG1000G-BF-C,13,AG1000G-BF-2,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,3.0,True
4,AG1000G-CD,76,AG1000G-CD,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,3.0,True
5,AG1000G-CF,73,AG1000G-CF,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,3.0,True
6,AG1000G-CI,80,AG1000G-CI,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,3.0,True
7,AG1000G-CM-A,303,AG1000G-CM-1,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,3.0,True
8,AG1000G-CM-B,97,AG1000G-CM-2,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,3.0,True
9,AG1000G-CM-C,44,AG1000G-CM-3,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,3.0,True


In [67]:
ag3.count_samples(
    sample_sets=['1272-VO-GM-OPONDO-VMF00160', 'AG1000G-CM-C', 'AG1000G-KE', 'AG1000G-ML-B', 'AG1000G-MW', 'AG1000G-TZ', 'AG1000G-UG', 'AG1000G-BF-B', '1191-VO-MULTI-OLOUGHLIN-VMF00140', '1246-VO-TZ-KABULA-VMF00185', '1246-VO-TZ-KABULA-VMF00197'],
    sample_query='country == ["Gambia, The", "Tanzania", "Burkina Faso", "Cameroon", "Kenya", "Mali", "Malawi", "Uganda"]',
    index=['country', 'year'])

taxon              arabiensis  bissau  coluzzii  gambiae  pwani  unassigned
country      year                                                          
Burkina Faso 2014          12       0       143      140      0           2
             2015          88       0       183      140      0           1
             2016          14       0       167      104      0           1
             2017           1       0       164       37      0           0
Cameroon     2013           2       0        19       23      0           0
Gambia, The  2001           1       0         0        0      0           0
             2019         314       6        37       19      0           5
Kenya        2000           0       0         0       19      0           0
             2007           3       0         0        0      0           0
             2012          10       0         0        0     54           0
Malawi       2015          41       0         0        0      0           0
Mali         2004           2       0        36       33      0           0
Tanzania     2012          87       0         0        0      0           0
             2013           1       0         0       32     10           0
             2015         137       0         0       32      1           0
             2018         467       0         0        0      0           0
             2019          97       0         0      227      0           3
Uganda       2012          82       0         0      207      0           1

In [3]:
region = "3L"
n_snps = 100_000

pca_df_arab_refs_pub, evr_arab_refs_pub = ag3.pca(
    region=region,
    n_snps=n_snps,
    sample_sets=['1272-VO-GM-OPONDO-VMF00160', 'AG1000G-CM-C', 'AG1000G-KE', 'AG1000G-ML-B', 'AG1000G-MW', 'AG1000G-TZ', 'AG1000G-UG', 'AG1000G-BF-B', '1191-VO-MULTI-OLOUGHLIN-VMF00140', '1246-VO-TZ-KABULA-VMF00185', '1246-VO-TZ-KABULA-VMF00197'],
    sample_query=f"taxon == 'arabiensis' and country == ['Gambia, The', 'Tanzania', 'Burkina Faso', 'Cameroon', 'Kenya', 'Mali', 'Malawi', 'Uganda'] and sample_id not in {ref_arab_outliers}",
    site_mask="arab",
    min_minor_ac=2,
    max_missing_an=0
    )

In [3]:
ag3.plot_pca_variance(evr_arab_refs_pub)

In [ ]:
ag3.plot_pca_coords(
    pca_df_arab_refs_pub,
    color='country',
    color_discrete_sequence=px.colors.qualitative.Dark24,
    width=850,
    title = "Geographical population structure in An. arabiensis in The Gambia + reference populations"
)

**Remove outliers and top up to the ref outliers list**

In [ ]:
ref_arab_outliers_pub = [
    "VBS48175-6366STDY9854776",
    "VBS47997-6366STDY9854595",
    "VBS48073-6366STDY9854673",
    "VBS48085-6366STDY9854685",
    "VBS48029-6366STDY9854629",
    "VBS47995-6366STDY9854593",
    "VBS48055-6366STDY9854655",
    "VBS47767-6366STDY9854376",
    "VBS48013-6366STDY9854611",
    "VBS48131-6366STDY9854733",
    "VBS47999-6366STDY9854597",
    "VBS48028-6366STDY9854628",
    "VBS48133-6366STDY9854735",
    "VBS48058-6366STDY9854658",
    "VBS58980-6781STDY12201453", #addition from here
    "VBS59067-6781STDY12201542",
    "VBS59065-6781STDY12201540",
    "VBS59074-6781STDY12201549",
    "VBS59101-6781STDY12201576",
    "VBS59098-6781STDY12201573",
    "VBS59007-6781STDY12201480",
    "VBS59004-6781STDY12201477"
]

In [ ]:
region = "3L"
n_snps = 100_000

pca_df_arab_refs_pub2, evr_arab_refs_pub2 = ag3.pca(
    region=region,
    n_snps=n_snps,
    sample_sets=['1272-VO-GM-OPONDO-VMF00160', 'AG1000G-CM-C', 'AG1000G-KE', 'AG1000G-ML-B', 'AG1000G-MW', 'AG1000G-TZ', 'AG1000G-UG', 'AG1000G-BF-B', '1191-VO-MULTI-OLOUGHLIN-VMF00140', '1246-VO-TZ-KABULA-VMF00185', '1246-VO-TZ-KABULA-VMF00197'],
    sample_query=f"taxon == 'arabiensis' and country == ['Gambia, The', 'Tanzania', 'Burkina Faso', 'Cameroon', 'Kenya', 'Mali', 'Malawi', 'Uganda'] and sample_id not in {ref_arab_outliers_pub}",
    site_mask="arab",
    min_minor_ac=2,
    max_missing_an=0
    )

In [ ]:
ag3.plot_pca_variance(evr_arab_refs_pub2)

In [ ]:
ag3.plot_pca_coords(
    pca_df_arab_refs_pub2,
    color='country',
    color_discrete_sequence=px.colors.qualitative.Dark24,
    width=850,
    title = "Geographical population structure in An. arabiensis in The Gambia + reference populations"
)

In [ ]:
region = "3L"
n_snps = 100_000

pca_df_arab_refs_pub3, evr_arab_refs_pub3 = ag3.pca(
    region=region,
    n_snps=n_snps,
    sample_sets=['1272-VO-GM-OPONDO-VMF00160', 'AG1000G-CM-C', 'AG1000G-ML-B', 'AG1000G-BF-B', '1191-VO-MULTI-OLOUGHLIN-VMF00140'],
    sample_query=f"taxon == 'arabiensis' and country == ['Gambia, The', 'Burkina Faso', 'Cameroon', 'Mali', 'Malawi'] and sample_id not in {ref_arab_outliers_pub}",
    site_mask="arab",
    min_minor_ac=2,
    max_missing_an=0
    )

In [ ]:
ag3.plot_pca_variance(evr_arab_refs_pub3)

In [ ]:
ag3.plot_pca_coords(
    pca_df_arab_refs_pub3,
    color='country',
    color_discrete_sequence=px.colors.qualitative.Dark24,
    width=850,
    title = "Geographical population structure in An. arabiensis in The Gambia + reference populations"
)

In [ ]:
region = "3L"
n_snps = 100_000

pca_df_arab_refs_pub4, evr_arab_refs_pub4 = ag3.pca(
    region=region,
    n_snps=n_snps,
    sample_sets=['1272-VO-GM-OPONDO-VMF00160', 'AG1000G-CM-C', 'AG1000G-ML-B', 'AG1000G-BF-B'],
    sample_query=f"taxon == 'arabiensis' and country == ['Gambia, The', 'Burkina Faso', 'Cameroon', 'Mali'] and sample_id not in {ref_arab_outliers_pub}",
    site_mask="arab",
    min_minor_ac=2,
    max_missing_an=0
    )

In [ ]:
ag3.plot_pca_variance(evr_arab_refs_pub4)

In [ ]:
color_mapping_admin2_refs = {
    'Jarra East': '#8e8c84',
    'Lower Fuladu West': '#7f7f7f',
    'Upper Fuladu West': '#6b705c',
    'Niani': '#4a4e4d',
    'Niamina East': '#3b5c40',
    'Jimara': '#8e7d32',
    'Basse': '#a95b29',
    'Wuli East': '#b64d3b',
    'Tumana': '#8a4b39',
    'Niamina West': '#6f4e37',
    'Sandu': '#3e3a3a',
    'Jeshwang': '#a4c9e2',
    'Kombo Central': '#78aaff',
    'Lower Niumi': '#4d88ff',
    'Bakau': '#0077b6',
    'Illiasa': '#0096c7',
    'Foni Brefet': '#00b4d8',
    'Houet': '#ff0072',
    'Benoue': '#e673af',
    'Bla': '#c798ef',
    'Baroueli': '#c798ef'
}

### **Supplementary Figure 3. Population structure - The Gambia and reference populations WA**

In [ ]:
ag3.plot_pca_coords(
    pca_df_arab_refs_pub4,
    color='admin2_name',
    color_discrete_map=color_mapping_admin2_refs,
    width=960,
    height=650,
    title = "Geographical population structure in An. arabiensis in The Gambia + Ag3.0 Burkina Faso, Mali, Cameroon",
)

### **Pairwise Average Fst between coastal and inland cohorts**

Compute pairwise average Hudson’s Fst between a set of specified cohorts

### **Figure 1c. Fst analysis**

In [ ]:
#define some useful variables

#lists of chromosome/arms for analyses
chrom_arms = ['2R', '2L', '3R','3L','X']
chroms = ['2RL', '3RL','X']

#dict of  analysis groupings
admin1_dict = {
    'Basse' : "admin2_name == 'Basse' | admin2_name == 'Jimara' | admin2_name == 'Sandu' | admin2_name == 'Tumana' | admin2_name == 'Wuli East'",
    'Brikama' : "admin2_name == 'Foni Brefet' | admin2_name == 'Kombo Central'",
    'Janjanbureh' : "admin2_name == 'Lower Fuladu West' | admin2_name == 'Niamina East' | admin2_name == 'Niamina West' | admin2_name == 'Upper Fuladu West'",
    'Kanifing' : "admin2_name == 'Bakau' | admin2_name == 'Jeshwang'",
    'Kerewan' : "admin2_name ==  'Illiasa' | admin2_name == 'Lower Niumi'",
    'Kuntaur' : "admin2_name == 'Niani'",
    'Mansakonko' : "admin2_name == 'Jarra East'"
}

admin2_dict = {
    'Basse' : "location == 'Kaba Kamma' | location == 'Sabi' | location == 'Santo Su'",
    'Jimara' : "location == 'Damba kunda Kuta' | location == 'Gambisara' | location == 'Kossemar' | location == 'Sotuma sere'",
    'Sandu' : "location == 'Sare Demba'",
    'Tumana' : "location == 'Nyakoi Madina' | location == 'Sare Alpha' | location == 'Tambansang' | location == 'Tuba Simoto'",
    'Wuli East' : "location == 'Baja Kunda'",
    'Foni Brefet' : "location == 'Bullock'",
    'Kombo Central' : "location == 'Brikama'",
    'Lower Fuladu West' : "location == 'Boiram' | location == 'Jakhali' | location == 'Kerewan' | location == 'Sankulay kunda' | location == 'Sapu' | location == 'Saruja' | location == 'Wali Kunda'",
    'Niamina East' : "location== 'Jareng' | location == 'Sare Silli'",
    'Niamina West' : "location == 'Kumbija'",
    'Upper Fuladu West' : "location == 'Bansang'",
    'Bakau' : "location == 'Bakau'",
    'Jeshwang' : "location == 'Ebotown', location == 'Tallinding'",
    'Illiasa' : "location ==  'Farafenni'",
    'Lower Niumi' : "location ==  'Essau'",
    'Niani' : "location == 'Kuntaur' | location == 'Nyanga Batang'",
    'Jarra East' : "location == 'Dongoroba'"
}

#dict of cohorts/locations for labelling
cohort_dict = {
    'GM-U_arab_2019' : 'Basse',
    'GM-W_arab_2019' : 'Brikama',
    'GM-M_arab_2019' : 'Janjanbureh',
    'GM-W_arab_2019' : 'Kanifing',
    'GM-N_arab_2019' : 'Kerewan',
    'GM-M_arab_2019' : 'Kuntaur',
    'GM-L_arab_2001' : 'Mansakonko',
    'GM-L_arab_2019' : 'Mansakonko'
}

cohort_dict2 = {
    'GM-U_arab_2019' : 'Basse',
    'GM-U_arab_2019' : 'Jimara',
    'GM-U_arab_2019' : 'Sandu',
    'GM-U_arab_2019' : 'Tumana',
    'GM-U_arab_2019' : 'Wuli East',
    'GM-W_arab_2019' : 'Foni Brefet',
    'GM-W_arab_2019' : 'Kombo Central',
    'GM-M_arab_2019' : 'Lower Fuladu West',
    'GM-M_arab_2019' : 'Niamina East',
    'GM-M_arab_2019' : 'Niamina West',
    'GM-M_arab_2019' : 'Upper Fuladu West',
    'GM-W_arab_2019' : 'Bakau',
    'GM-W_arab_2019' : 'Jeshwang',
    'GM-N_arab_2019' : 'Illiasa',
    'GM-N_arab_2019' : 'Lower Niumi',
    'GM-M_arab_2019' : 'Niani',
    'GM-L_arab_2019' : 'Jarra East'
}

In [ ]:
fst_arab_df = ag3.pairwise_average_fst(
    region="3L:15,000,000-41,000,000",
    cohorts="admin2_name",
    sample_query="country == 'Gambia, The' and contributor not in ['Alfred Amambua-Ngwa'] and taxon == 'arabiensis'",
    min_cohort_size=10,
    site_mask="arab",
)
fst_arab_df

In [ ]:
ag3.plot_pairwise_average_fst(fst_arab_df)

In [ ]:
ag3.plot_pairwise_average_fst(fst_arab_df)

In [ ]:
ag3.plot_pairwise_average_fst(fst_arab_df)

In [ ]:
fst_arab_df2 = ag3.pairwise_average_fst(
    region="3L",
    cohorts="admin1_name",
    sample_query="country == 'Gambia, The' and sample_set == '1272-VO-GM-OPONDO-VMF00160' and taxon == 'arabiensis'",
    min_cohort_size=10,
    site_mask="arab",
)
fst_arab_df2

In [ ]:
ag3.plot_pairwise_average_fst(fst_arab_df2)

In [ ]:
#Fst between cohorts of admin1
import itertools as it
chroms = ['2RL','3RL','X']
admin1_locs = ['Basse', 'Brikama', 'Janjanbureh', 'Kanifing', 'Kerewan', 'Kuntaur', 'Mansakonko']

fst_df_list = []
for region_a,region_b in it.combinations(admin1_locs,2):
    for chrom in chroms:
        fstar = ag3.fst_gwss(contig = f'{chrom}', min_cohort_size=10, window_size=5000, cohort1_query=admin1_dict[f'{region_a}'], cohort2_query=admin1_dict[f'{region_b}'])
        fst_df = pd.DataFrame({'pos':fstar[0], 'fst':fstar[1]})
        fst_df['chrom'] = chrom
        fst_df['admin1_a'] = region_a
        fst_df['admin1_b'] = region_b
        fst_df_list.append(fst_df)

    fst_bigdf = pd.concat(fst_df_list)

#fst_bigdf.to_csv('./Fst-analysis-results/fstdf_admin1.csv')